# RAG-песочница: эксперименты без правок кода

Ноутбук для экспериментов с готовым RAG-пайплайном vue_rag.

**Что делает:**

- ищет ответ в уже построенной базе: дамп -> БД -> поиск -> топ-N доказательств;
- по желанию генерирует ответ моделью (по умолчанию выключено);
- показывает текущие чанки документа и превью пересчёта с другими параметрами;
- сохраняет артефакты прогона в `runtime/debug/experiments/`.

**Код проекта не переписывается:** ноутбук вызывает те же классы, что и боевой
пайплайн (`SearchService`, `ChatService`, `LadybugAnn`, `OllamaAdapter`). Если код
mvp_v2 поменялся - просто снова выполните Run All: ячейка импортов перезагружает
модули в правильном порядке (контракты до сервисов).

**Как пользоваться:**

1. Выберите ядро основного окружения проекта (`mvp_v2/.venv`).
2. Настройки меняйте **только в ячейке CONFIG**.
3. Выполните Run All.

Всё остальное - read-only: дамп, БД и снимок не изменяются; записи только в
`runtime/debug/experiments/` (JSON-артефакты прогонов).

## 1. Корень проекта и ядро

Проверяем, что ноутбук запущен во внешнем окружении проекта
`mvp_v2/.venv` (основное окружение проекта), находим корень репозитория
и подключаем `.env`.

In [6]:
from __future__ import annotations

import importlib
import json
import os
import sys
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/vue_rag").is_dir():
            return candidate
        nested = candidate / "mvp_v2"
        if (nested / "pyproject.toml").is_file() and (nested / "src/vue_rag").is_dir():
            return nested
    raise RuntimeError("Run this notebook from the repository or mvp_v2 directory")


PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)
print("Project:", PROJECT_ROOT)
print("Kernel:", sys.executable)

Project: /Users/zero/GITHUB/vue-graph-rag/mvp_v2
Kernel: /Users/zero/GITHUB/vue-graph-rag/mvp_v2/.venv/bin/python


## 2. CONFIG - единственная ячейка настроек

Корни дампов и баз берутся из `.env`: `MVP_V2_DUMPS_ROOT`, `MVP_V2_DATABASES_ROOT`,
`MVP_V2_WORKING_ROOT`, `MVP_V2_LADYBUG_HOME` (по умолчанию - `mvp_v2/runtime/...`).
Ollama задаётся `MVP_V2_OLLAMA_BASE_URL` (и опционально
`MVP_V2_OLLAMA_ALLOWED_LAN_HOSTS`), модель генерации - `MVP_V2_CHAT_MODEL` и
`MVP_V2_CHAT_MODEL_VERSION`.

In [7]:
# ===================== CONFIG: меняйте только здесь =====================

# База данных: None = самая свежая проверенная .lbdb из runtime/databases,
# иначе имя базы без суффикса, например "rag-db".
DATABASE_NAME: str | None = None

# Вопросы для экспериментов (замените на свои).
MY_QUESTIONS: tuple[str, ...] = (
    "Что нужно для того, чтобы мы начали работу над твоим резюме?",
    "Где купить прокси?",
    "Сколько рабочих почт, номеров, SIM-карт, HH и Telegram нужно создать?",
)

EVIDENCE_LIMIT = 3               # топ-N доказательств (1..100)
MINIMUM_SIMILARITY_SCORE = 0.55  # порог косинус-близости (-1..1)

# Песочница чанкинга (раздел 5): только просмотр, ничего не пишет.
CHUNK_DOCUMENT_PATH = ""         # подстрока local_path; "" = первый .md-документ
PREVIEW_CHARACTERS = 300         # сколько символов текста показывать
SANDBOX_MAX_CHARACTERS = 1_200   # параметры политики для превью пересчёта
SANDBOX_OVERLAP_CHARACTERS = 120

# Генерация ответов моделью (True = звать Ollama-генерацию; дольше и дороже).
GENERATE_ANSWERS = True
PROMPT_PROFILE_NAME = "support-v1"
GENERATION_TEMPERATURE = 0.0
CHAT_MODEL = os.environ.get("MVP_V2_CHAT_MODEL", "")
CHAT_MODEL_VERSION = os.environ.get("MVP_V2_CHAT_MODEL_VERSION", "")
OLLAMA_TIMEOUT_SECONDS = 120.0

# Куда сохранять артефакты прогона (run.json, results.json, answers.json).
RUNS_ROOT = PROJECT_ROOT / "runtime/debug/experiments"

# --- проверки значений ---
if not MY_QUESTIONS or any(not question.strip() for question in MY_QUESTIONS):
    raise ValueError("MY_QUESTIONS: нужен хотя бы один непустой вопрос")
if not 1 <= EVIDENCE_LIMIT <= 100:
    raise ValueError("EVIDENCE_LIMIT должен быть от 1 до 100")
if not -1.0 <= MINIMUM_SIMILARITY_SCORE <= 1.0:
    raise ValueError("MINIMUM_SIMILARITY_SCORE должен быть в диапазоне [-1, 1]")
if GENERATE_ANSWERS and (not CHAT_MODEL.strip() or not CHAT_MODEL_VERSION.strip()):
    raise ValueError(
        "Для GENERATE_ANSWERS задайте MVP_V2_CHAT_MODEL и "
        "MVP_V2_CHAT_MODEL_VERSION в .env"
    )
print(
    "CONFIG ok:",
    len(MY_QUESTIONS), "вопросов; лимит", EVIDENCE_LIMIT,
    "; порог", MINIMUM_SIMILARITY_SCORE,
    "; генерация", "вкл" if GENERATE_ANSWERS else "выкл",
)

CONFIG ok: 3 вопросов; лимит 3 ; порог 0.55 ; генерация выкл


## 3. Импорты и reload (контракты до сервисов)

Ячейка импортирует модули mvp_v2 и перезагружает их по порядку зависимостей:
контракты/DTO -> домен -> подготовка -> адаптеры -> сервисы -> компоновка. Так в
живом ядре не остаются устаревшие Pydantic-классы после правок кода. Имена
перепривязываются из свежих модулей.

In [8]:
# Контракты (DTO) перезагружаются ДО сервисов: иначе в живом ядре остаются
# устаревшие Pydantic-классы. После правок кода mvp_v2 просто снова Run All.
import vue_rag.application.ingestion_contracts as ingestion_contracts_module
import vue_rag.application.dto.retrieval as retrieval_dto_module
import vue_rag.application.dto.chat as chat_dto_module
import vue_rag.application.dump_lifecycle.manifests as manifests_module
import vue_rag.domain.snapshot as snapshot_module
import vue_rag.application.rag.preparation.chunking as chunking_module
import vue_rag.application.rag.preparation.chunk_backfill as chunk_backfill_module
import vue_rag.adapters.filesystem.media as media_module
import vue_rag.adapters.ladybug as ladybug_adapter_module
import vue_rag.adapters.ollama as ollama_module
import vue_rag.application.rag.agent.prompts as prompts_module
import vue_rag.application.rag.agent.answer_rendering as rendering_module
import vue_rag.application.rag.retrieval.retrieval as retrieval_module
import vue_rag.application.rag.agent.chat as chat_module
import vue_rag.bootstrap.ui as bootstrap_ui_module
import vue_rag.streamlit_ui.controller as controller_module

_RELOAD_ORDER = (
    ingestion_contracts_module,  # контракты
    retrieval_dto_module,
    chat_dto_module,
    manifests_module,
    snapshot_module,             # домен
    chunking_module,             # подготовка
    chunk_backfill_module,
    media_module,                # адаптеры
    ladybug_adapter_module,
    ollama_module,
    prompts_module,              # сервисы
    rendering_module,
    retrieval_module,
    chat_module,
    bootstrap_ui_module,         # компоновка (зависит от адаптеров)
    controller_module,
)
for module in _RELOAD_ORDER:
    importlib.reload(module)

# Свежие имена из перезагруженных модулей.
ModelAuthorization = ingestion_contracts_module.ModelAuthorization
SearchRequest = retrieval_dto_module.SearchRequest
EmbeddingPurpose = retrieval_dto_module.EmbeddingPurpose
VectorHit = retrieval_dto_module.VectorHit
VectorQuery = retrieval_dto_module.VectorQuery
ChatRequest = chat_dto_module.ChatRequest
GraphSnapshot = snapshot_module.GraphSnapshot
ChunkPolicy = chunking_module.ChunkPolicy
split_text = chunking_module.split_text
sanitize_source_text = chunk_backfill_module._sanitize_source_text
FilesystemMediaResolver = media_module.FilesystemMediaResolver
load_verified_dump = ladybug_adapter_module.load_verified_dump
LadybugAnn = ladybug_adapter_module.LadybugAnn
LocalHttpTransport = ollama_module.LocalHttpTransport
OllamaAdapter = ollama_module.OllamaAdapter
PROMPT_TEMPLATES = prompts_module.PROMPT_TEMPLATES
SearchService = retrieval_module.SearchService
ChatService = chat_module.ChatService
default_runtime_config = bootstrap_ui_module.default_runtime_config
configure_ladybug_home = controller_module.configure_ladybug_home
scan_database_manifests = controller_module.scan_database_manifests
database_is_stale = controller_module.database_is_stale

# Защита от устаревших типов в живом ядре.
if "minimum_score" not in SearchRequest.model_fields:
    raise RuntimeError(
        "SearchRequest устарел: перезапустите ядро и выполните Run All"
    )
if GENERATE_ANSWERS and PROMPT_PROFILE_NAME not in PROMPT_TEMPLATES:
    available = ", ".join(sorted(PROMPT_TEMPLATES))
    raise ValueError(
        f"Неизвестный PROMPT_PROFILE_NAME {PROMPT_PROFILE_NAME!r}; доступные: {available}"
    )
print("Модули загружены и перезагружены в порядке зависимостей")

Модули загружены и перезагружены в порядке зависимостей


## 4. Загрузка БД и исходного дампа (read-only)

Выбираем проверенную базу из `runtime/databases` (по умолчанию - новейшую),
подгружаем соответствующий дамп из `runtime/dumps` и сверяем их манифесты.

In [9]:
config = default_runtime_config()
verified_databases = scan_database_manifests(config.databases_root)
if not verified_databases:
    raise RuntimeError(
        "Проверенные БД не найдены в "
        + str(config.databases_root)
        + "; постройте базу через Streamlit или Telegram"
    )

if DATABASE_NAME is None:
    database_path, database_manifest = max(
        verified_databases, key=lambda item: item[1].built_at
    )
    print("DATABASE_NAME = None -> выбрана новейшая БД по built_at")
else:
    wanted = (
        DATABASE_NAME if DATABASE_NAME.endswith(".lbdb") else DATABASE_NAME + ".lbdb"
    )
    selected = [item for item in verified_databases if item[0].name == wanted]
    if len(selected) != 1:
        raise RuntimeError(
            f"Проверенная БД {wanted!r} не найдена однозначно в {config.databases_root}"
        )
    database_path, database_manifest = selected[0]

source_dump_path = (
    config.dumps_root / f"dump-{database_manifest.source_dump_number:06d}"
)
dump_manifest, snapshot = load_verified_dump(source_dump_path)
if database_is_stale(database_manifest, dump_manifest):
    raise RuntimeError("Манифест БД не совпадает с исходным дампом")

print("Database   :", database_path.name)
print("Source dump:", source_dump_path.name, "run", dump_manifest.run_id)
print("Space      :", database_manifest.embedding_space.model_dump(mode="json"))
print(
    "Snapshot   : документов", len(snapshot.documents),
    "| версий", len(snapshot.document_versions),
    "| чанков", len(snapshot.chunks),
    "| эмбеддингов", len(snapshot.embeddings),
)
print("Дальше всё read-only: дамп, снимок и БД не изменяются")

DATABASE_NAME = None -> выбрана новейшая БД по built_at
Database   : clean-v9-onboarding.lbdb
Source dump: dump-000058 run 0863af4aacf247965ae5ff2fbcfd3563f1033e8c05c6a59b0b162b5772d20456
Space      : {'provider': 'ollama', 'model': 'bge-m3', 'model_version': '790764642607', 'dimension': 1024}
Snapshot   : документов 30 | версий 33 | чанков 209 | эмбеддингов 223
Дальше всё read-only: дамп, снимок и БД не изменяются


## 5. Песочница чанкинга (только просмотр)

Показывает текущие чанки выбранного документа из дампа (позиция, длина, границы,
перекрытие соседей) и превью пересчёта с параметрами `SANDBOX_*`. Ничего не пишет.

In [10]:
resolver = FilesystemMediaResolver(config.working_root)

current_md_versions = sorted(
    (
        version
        for version in snapshot.document_versions
        if version.is_current
        and version.local_path
        and version.local_path.endswith(".md")
    ),
    key=lambda version: version.local_path or "",
)
if not current_md_versions:
    raise RuntimeError("В дампе нет текущих .md-версий документов")

if CHUNK_DOCUMENT_PATH:
    candidates = [
        version
        for version in current_md_versions
        if CHUNK_DOCUMENT_PATH in (version.local_path or "")
    ]
else:
    candidates = current_md_versions[:1]
if len(candidates) != 1:
    raise ValueError(
        f"CHUNK_DOCUMENT_PATH выбирает {len(candidates)} документов (нужен ровно один)"
    )
version = candidates[0]

document_id = next(
    (edge.source_id for edge in snapshot.has_versions if edge.target_id == version.id),
    None,
)
title = next(
    (document.title for document in snapshot.documents if document.id == document_id),
    version.local_path or "?",
)

current_chunks = sorted(
    (
        chunk
        for chunk in snapshot.chunks
        if any(
            edge.source_id == version.id and edge.target_id == chunk.id
            for edge in snapshot.has_chunks
        )
    ),
    key=lambda chunk: chunk.position,
)

print(f"Документ: {title!r}")
print(f"Версия {version.version}, путь: {version.local_path}")
print(f"Чанков сейчас: {len(current_chunks)}")
for chunk in current_chunks:
    head = chunk.content[:60].replace("\n", "\\n")
    tail = chunk.content[-40:].replace("\n", "\\n")
    print(f"  [{chunk.position:02d}] len={len(chunk.content):4d}  {head} ... {tail}")

for previous, following in zip(current_chunks, current_chunks[1:]):
    overlap = ""
    limit = min(len(previous.content), len(following.content), 120)
    for size in range(limit, 0, -1):
        if previous.content.endswith(following.content[:size]):
            overlap = following.content[:size]
            break
    print(f"  overlap {previous.position}->{following.position}: {len(overlap)} симв.")

raw_text = resolver.read_verified(version.local_path, version.sha256).decode("utf-8")
sanitized_text = sanitize_source_text(raw_text)
if sanitized_text != raw_text:
    print(
        "ВНИМАНИЕ: в исходнике есть control-символы. Инжест чанкует сырой текст, "
        "backfill - санитизированный (баг-кандидат, docs/code-review/02-stage-chunking.md)."
    )

sandbox_policy = ChunkPolicy(
    max_characters=SANDBOX_MAX_CHARACTERS,
    overlap_characters=SANDBOX_OVERLAP_CHARACTERS,
)
sandbox_chunks = split_text(sanitized_text, sandbox_policy)
print(
    f"Превью пересчёта (max={SANDBOX_MAX_CHARACTERS}, "
    f"overlap={SANDBOX_OVERLAP_CHARACTERS}): {len(sandbox_chunks)} чанков "
    f"против текущих {len(current_chunks)}"
)
for position, piece in enumerate(sandbox_chunks[:10]):
    print(f"  [{position:02d}] len={len(piece):4d}  {piece[:PREVIEW_CHARACTERS]!r}")
if len(sandbox_chunks) > 10:
    print(f"  ... и ещё {len(sandbox_chunks) - 10} чанков")
print("Ничего не сохраняется: это превью в ноутбуке, дамп и БД не меняются")

Документ: 'Чеклист легенды (для Киры)'
Версия 1, путь: documents/doc-09ee24d8260e884437e55180.md
Чанков сейчас: 1
  [00] len= 347  # Чеклист легенды (для Киры)\n\n- образование\n- возраст\n- поче ... ие\n- место жительства\n- что хочет сейчас
Превью пересчёта (max=1200, overlap=120): 1 чанков против текущих 1
  [00] len= 347  '# Чеклист легенды (для Киры)\n\n- образование\n- возраст\n- почему пришел в первую команду\n- почему ушел из первой компании\n- почему уходит из последней\n- в каких командах работал и какой продукт пилил\n- годы работы на работах (кол-во лет опыта)\n- разные домены работы\n- основные достижения\n- скилл сет\n-'
Ничего не сохраняется: это превью в ноутбуке, дамп и БД не меняются


## 6. Поиск по моим вопросам (read-only)

Для каждого вопроса: эмбеддинг вопроса через Ollama -> ANN-поиск в Ladybug ->
фильтрация по порогу `MINIMUM_SIMILARITY_SCORE` -> гидрация доказательств из
снимка. Печатаются и сырые ANN-кандидаты (звёздочкой помечены прошедшие порог),
и финальные доказательства. База открывается в режиме read-only.

In [11]:
ollama_base_url = os.environ.get("MVP_V2_OLLAMA_BASE_URL", "").strip()
if not ollama_base_url:
    raise RuntimeError(
        "MVP_V2_OLLAMA_BASE_URL не задан в .env: эмбеддинг вопроса невозможен"
    )
allowed_lan_hosts = frozenset(
    host.strip()
    for host in os.environ.get("MVP_V2_OLLAMA_ALLOWED_LAN_HOSTS", "").split(",")
    if host.strip()
)

embedding_model = OllamaAdapter(
    LocalHttpTransport(
        ollama_base_url,
        timeout_seconds=OLLAMA_TIMEOUT_SECONDS,
        allowed_lan_hosts=allowed_lan_hosts,
    ),
    generation_temperature=GENERATION_TEMPERATURE,
)


class RecordingAnn:
    """Обёртка над ANN-портом: запоминает сырых кандидатов до фильтров."""

    def __init__(self, ann: Any) -> None:
        self._ann = ann
        self.hits: tuple[VectorHit, ...] = ()

    def query(self, request: VectorQuery) -> tuple[VectorHit, ...]:
        self.hits = self._ann.query(request)
        return self.hits


EXPERIMENT_RESULTS: list[dict[str, Any]] = []
EXPERIMENT_STARTED_AT = datetime.now(UTC)

configure_ladybug_home(config)
ladybug_module = importlib.import_module("ladybug")
database = ladybug_module.Database(str(database_path / "database.lbug"), read_only=True)
try:
    warmup_connection = ladybug_module.Connection(database)
    try:
        warmup_result = warmup_connection.execute("LOAD vector")
        warmup_result.close()
    finally:
        warmup_connection.close()

    for index, question in enumerate(MY_QUESTIONS, start=1):
        connection = ladybug_module.Connection(database)
        recorder: RecordingAnn | None = None
        evidence: tuple = ()
        failure_info: dict[str, str] | None = None
        started_at = time.perf_counter()
        try:
            recorder = RecordingAnn(LadybugAnn(connection, database_manifest))
            service = SearchService(
                embedding_model, recorder, database_manifest, snapshot
            )
            request = SearchRequest(
                question=question,
                space=database_manifest.embedding_space,
                purposes=(
                    EmbeddingPurpose.CHUNK_RETRIEVAL,
                    EmbeddingPurpose.ASSET_DESCRIPTION_RETRIEVAL,
                ),
                limit=EVIDENCE_LIMIT,
                minimum_score=MINIMUM_SIMILARITY_SCORE,
                authorization=ModelAuthorization(allow=True, operation="embedding"),
            )
            evidence = service.search(request)
        except Exception as failure:  # noqa: BLE001 - показываем и не роняем прогон
            failure_info = {"type": type(failure).__name__, "message": str(failure)}
        finally:
            connection.close()
        latency_ms = round((time.perf_counter() - started_at) * 1_000, 3)

        print(f"--- [{index}/{len(MY_QUESTIONS)}] {question}")
        if failure_info is not None:
            print(f"    ERROR {failure_info['type']}: {failure_info['message']}")
        else:
            raw_hits = recorder.hits if recorder is not None else ()
            kept_ids = {item.embedding_id for item in evidence}
            print(
                f"    ANN-кандидатов: {len(raw_hits)}; после порога "
                f"{MINIMUM_SIMILARITY_SCORE} и дедупа: {len(evidence)}; {latency_ms} мс"
            )
            for hit in raw_hits[: EVIDENCE_LIMIT + 2]:
                kept = "*" if hit.embedding_id in kept_ids else " "
                print(
                    f"    {kept} score={hit.score:.4f} {hit.subject.kind:20s} "
                    f"{hit.subject.id[:24]}"
                )
            for item in evidence:
                position = (
                    item.matched_chunk_position + 1
                    if item.matched_chunk_position is not None
                    else "-"
                )
                print(
                    f"    => score={item.score:.4f} [{item.subject.kind}] "
                    f"{item.document_title!r} chunk {position}/{item.document_chunk_count} "
                    f"scope={item.context_scope}"
                )
                print(f"       text: {item.text[:PREVIEW_CHARACTERS]!r}")

        EXPERIMENT_RESULTS.append(
            {
                "index": index,
                "question": question,
                "ann_hits": [
                    hit.model_dump(mode="json")
                    for hit in (recorder.hits if recorder is not None else ())
                ],
                "evidence": [item.model_dump(mode="json") for item in evidence],
                "latency_ms": latency_ms,
                "error": failure_info,
                "recorded_at": datetime.now(UTC).isoformat(),
            }
        )
finally:
    database.close()

print(f"\nПрогон завершён: {len(EXPERIMENT_RESULTS)} вопросов; соединения закрыты")

--- [1/3] Что нужно для того, чтобы мы начали работу над твоим резюме?
    ANN-кандидатов: 9; после порога 0.55 и дедупа: 3; 378.679 мс
    * score=0.6572 chunk                chunk-9f97a574b50b706cda
    * score=0.6524 chunk                chunk-af21446f01191a93e4
      score=0.6275 chunk                chunk-31ab74c15e05ce23e1
    * score=0.6274 chunk                chunk-06063f3a20a3168f8e
      score=0.6259 chunk                chunk-85659e3971c965b91d
    => score=0.6572 [chunk] 'Учебник по софт - скиллам' chunk 2/10 scope=chunk_window
       text: 'жит у тебя в папке “Самопрезентация” в твоей папке ученика, чтобы не забыть важные нюансы для рассказа про свой опыт.** \n\nХорошая практика: после первой попытки самопрезентации - прислать файл на вычитку вашему софт - ментору, чтобы он подправил какие-то моменты/нюансы и вы могли от этого файла уже '
    => score=0.6524 [chunk] 'Учебник по софт - скиллам' chunk 2/9 scope=chunk_window
       text: 'адо запомнить по ней.\n\nСоветуем ср

## 7. Генерация ответов (опционально)

Включается `GENERATE_ANSWERS = True` в CONFIG. Требует `MVP_V2_CHAT_MODEL` и
`MVP_V2_CHAT_MODEL_VERSION` в `.env` и работающий Ollama-генератор. Поиск и
пороги применяются те же, что в разделе 6.

In [13]:
ANSWERS: list[dict[str, Any]] = []

if not GENERATE_ANSWERS:
    print("GENERATE_ANSWERS = False: генерация пропущена (включается в CONFIG)")
else:
    prompt_template = PROMPT_TEMPLATES[PROMPT_PROFILE_NAME]
    configure_ladybug_home(config)
    ladybug_module = importlib.import_module("ladybug")
    database = ladybug_module.Database(
        str(database_path / "database.lbug"), read_only=True
    )
    try:
        for index, question in enumerate(MY_QUESTIONS, start=1):
            connection = ladybug_module.Connection(database)
            try:
                service = ChatService(
                    SearchService(
                        embedding_model,
                        LadybugAnn(connection, database_manifest),
                        database_manifest,
                        snapshot,
                    ),
                    embedding_model,
                    prompt_template=prompt_template,
                )
                request = ChatRequest(
                    question=question,
                    search=SearchRequest(
                        question=question,
                        space=database_manifest.embedding_space,
                        purposes=(
                            EmbeddingPurpose.CHUNK_RETRIEVAL,
                            EmbeddingPurpose.ASSET_DESCRIPTION_RETRIEVAL,
                        ),
                        limit=EVIDENCE_LIMIT,
                        minimum_score=MINIMUM_SIMILARITY_SCORE,
                        authorization=ModelAuthorization(
                            allow=True, operation="embedding"
                        ),
                    ),
                    model=CHAT_MODEL.strip(),
                    model_version=CHAT_MODEL_VERSION.strip(),
                    authorization=ModelAuthorization(
                        allow=True, operation="generation"
                    ),
                )
                answer = service.answer(request)
            finally:
                connection.close()
            ANSWERS.append(
                {
                    "index": index,
                    "question": question,
                    "answer": answer.model_dump(mode="json"),
                }
            )
            print(f"--- [{index}/{len(MY_QUESTIONS)}] {question}")
            print("generated:", answer.generated)
            print(answer.answer)
            citations = ", ".join(
                f"[{citation.number}] {citation.document_title!r}"
                for citation in answer.citations
            )
            print("citations:", citations or "нет")
    finally:
        database.close()

GENERATE_ANSWERS = False: генерация пропущена (включается в CONFIG)


## 8. Сохранение артефактов прогона

`run.json` - параметры прогона; `results.json` - кандидаты и доказательства по
каждому вопросу; `answers.json` - сгенерированные ответы (если были). Одинаковые
вопросы при одинаковом коде должны давать одинаковые топ-N: удобно сверять прогоны
до и после экспериментов.

In [ ]:
run_id = EXPERIMENT_STARTED_AT.strftime("%Y%m%dT%H%M%S.%f") + "Z"
run_directory = RUNS_ROOT / EXPERIMENT_STARTED_AT.strftime("%Y-%m-%d") / run_id
run_directory.mkdir(parents=True, exist_ok=False)


def write_json_atomic(path: Path, payload: object) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


run_metadata = {
    "run_id": run_id,
    "notebook": "rag_experiments.ipynb",
    "started_at": EXPERIMENT_STARTED_AT.isoformat(),
    "saved_at": datetime.now(UTC).isoformat(),
    "config": {
        "database_name": database_path.name,
        "database_build_id": database_manifest.build_id,
        "source_dump_run_id": dump_manifest.run_id,
        "embedding_space": database_manifest.embedding_space.model_dump(mode="json"),
        "evidence_limit": EVIDENCE_LIMIT,
        "minimum_similarity_score": MINIMUM_SIMILARITY_SCORE,
        "generate_answers": GENERATE_ANSWERS,
        "prompt_profile_name": PROMPT_PROFILE_NAME,
        "questions": list(MY_QUESTIONS),
    },
    "case_count": len(EXPERIMENT_RESULTS),
}
write_json_atomic(run_directory / "run.json", run_metadata)
write_json_atomic(run_directory / "results.json", EXPERIMENT_RESULTS)
if ANSWERS:
    write_json_atomic(run_directory / "answers.json", ANSWERS)
print("Прогон сохранён:", run_directory)
print(
    "results.json можно сравнивать между прогонами: "
    "одни вопросы + один код -> один топ-N"
)

## Мои заметки

(записывайте сюда выводы из экспериментов)